# Egyptian Plate ANPR — fine-tune on real Egypt data (Colab GPU)

Runs the whole chain on the GPU:

1. Pull the **EALPR** benchmark (2,087 labelled Egyptian vehicles) — public on GitHub.
2. Rebuild it to **match this project's camera**, which is the step that decides
   whether the fine-tune is worth anything at all (see §4).
3. **Fine-tune** the plate detector on GPU.
4. Fit the plate-**colour** rule.
5. Measure against the real clip and write the **CSV**.

Runtime → Change runtime type → **T4 GPU** before starting.

## 1. GPU check

If this says `cpu`, stop — a fine-tune here takes ~18 min/epoch on CPU versus
~20 s/epoch on a T4.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch', torch.__version__)

## 2. Repo + dependencies

In [ ]:
import os, subprocess
REPO='https://github.com/yousseffbassemm/ITS-project-Elsewedy.git'
DIR='/content/its-traffic'
if os.path.isdir(DIR+'/.git'):
    subprocess.run(['git','-C',DIR,'fetch','--all'],check=False)
    subprocess.run(['git','-C',DIR,'checkout','plates-anpr'],check=False)
    subprocess.run(['git','-C',DIR,'pull','--ff-only'],check=False)
else:
    subprocess.run(['git','clone','-b','plates-anpr',REPO,DIR],check=True)
os.chdir(DIR)
print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. Get the data and the starting weights

EALPR is public on GitHub — no Kaggle or Roboflow credentials needed. ~280 MB.

In [ ]:
import pathlib, urllib.request, subprocess

if not pathlib.Path('data/plates/EALPR').exists():
    pathlib.Path('data/plates').mkdir(parents=True, exist_ok=True)
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ahmedramadan96/EALPR.git',
                    'data/plates/EALPR'], check=True)

v=len(list(pathlib.Path('data/plates/EALPR/EALPR Vechicles dataset/Vehicles').glob('*.jpg')))
p=len(list(pathlib.Path('data/plates/EALPR/EALPR- Plates dataset').glob('*.png')))
print(f'{v} labelled vehicles, {p} plate crops')

In [ ]:
import pathlib, urllib.request
pathlib.Path('models').mkdir(exist_ok=True)
WEIGHTS={
 'models/plate_detect.pt':'https://huggingface.co/morsetechlab/yolov11-license-plate-detection/resolve/main/license-plate-finetune-v1n.pt',
 'models/eg_alpr.pt':'https://huggingface.co/sshdopey/egyptian-license-plates/resolve/main/license_yolo_med_97.pt',
}
for d,u in WEIGHTS.items():
    if not pathlib.Path(d).exists(): urllib.request.urlretrieve(u,d)
    print(d, round(pathlib.Path(d).stat().st_size/1e6,1),'MB')

## 4. Rebuild the dataset to match the camera

**This is the step that makes or breaks the fine-tune.** Measured:

| | plate width | fraction of frame |
|---|---|---|
| EALPR (close-up photos) | median **136 px** | 0.173 |
| `street_egypt.mp4` (CCTV) | **20–35 px** | 0.027 |

EALPR plates are **4.5× larger** than any this model will meet in deployment.
Fine-tuning on the raw set optimises for objects that never occur, reports an
excellent val mAP, and changes nothing on the real footage — the most expensive
kind of wrong answer, because the metrics look like success.

So each image is rescaled until its plate lands in the deployment range and
composited onto a 1280×720 frame. 25% are kept native so the model kept its
close-up ability.

In [ ]:
!python -m tools.prep_plate_dataset --target-px 18 46 --native-frac 0.25

In [ ]:
# Look at the labels before training on them. A silent label bug is the
# single most common reason a detection fine-tune quietly fails.
import cv2, glob, random, numpy as np
import matplotlib.pyplot as plt
fs=sorted(glob.glob('data/plates/detect_ds/train/images/*.jpg'))
random.seed(3); tiles=[]
for f in random.sample(fs,6):
    im=cv2.imread(f); H,W=im.shape[:2]
    for line in open(f.replace('images','labels').replace('.jpg','.txt')):
        _,cx,cy,w,h=[float(v) for v in line.split()]
        x1,y1,x2,y2=int((cx-w/2)*W),int((cy-h/2)*H),int((cx+w/2)*W),int((cy+h/2)*H)
        cv2.rectangle(im,(x1,y1),(x2,y2),(0,0,255),2)
        cv2.putText(im,f'{x2-x1}px',(x1,max(y1-6,12)),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
    tiles.append(cv2.cvtColor(cv2.resize(im,(426,240)),cv2.COLOR_BGR2RGB))
fig,ax=plt.subplots(2,3,figsize=(16,6))
for a,t in zip(ax.ravel(),tiles): a.imshow(t); a.axis('off')
plt.tight_layout(); plt.show()

## 5. Fine-tune on the GPU

`fliplr=0` because a mirrored plate is not a thing this camera sees, and `scale`
is kept high because scale variation is the entire point of this run.

~1,660 train images, 60 epochs, T4: roughly **20–30 minutes**.

In [ ]:
from ultralytics import YOLO
import time, shutil, pathlib

t0=time.time()
m=YOLO('models/plate_detect.pt')
m.train(data='data/plates/detect_ds/data.yaml',
        epochs=60, imgsz=960, batch=16, device=0, workers=2,
        project='runs/plate', name='ealpr_ft', exist_ok=True,
        fliplr=0.0, flipud=0.0, degrees=5.0, mosaic=0.5, scale=0.6,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        patience=15, cos_lr=True, plots=True)
shutil.copy(m.trainer.best,'models/plate_detect_ft.pt')
print('trained in',round((time.time()-t0)/60,1),'min -> models/plate_detect_ft.pt')

In [ ]:
from ultralytics import YOLO
print('=== fine-tuned ==='); YOLO('models/plate_detect_ft.pt').val(data='data/plates/detect_ds/data.yaml', device=0)
print('=== baseline (before fine-tuning) ==='); YOLO('models/plate_detect.pt').val(data='data/plates/detect_ds/data.yaml', device=0)

## 6. The test that actually matters

val mAP is measured on synthesised frames. What decides whether this fine-tune
was worth doing is **recall on the real clip at 20–35 px**.

Upload `street_egypt.mp4` — it is gitignored (real CCTV of identifiable vehicles).

In [ ]:
import pathlib, os
clip=pathlib.Path('samples/street_egypt.mp4'); clip.parent.mkdir(exist_ok=True)
if not clip.exists():
    from google.colab import files
    up=files.upload()
    for n in up: os.replace(n,clip)
print('ready:',clip)

In [ ]:
import cv2, numpy as np
from ultralytics import YOLO

def probe(weights, frames=900, step=15):
    m=YOLO(weights); cap=cv2.VideoCapture('samples/street_egypt.mp4')
    ws=[]; hits=0; seen=0; i=0
    while i<frames:
        ok,f=cap.read()
        if not ok: break
        if i%step==0:
            seen+=1
            r=m.predict(f,imgsz=1280,conf=0.25,verbose=False,device=0)[0]
            if len(r.boxes): hits+=1
            ws+=[float(b[2]-b[0]) for b in r.boxes.xyxy.cpu().numpy()]
        i+=1
    cap.release()
    return dict(frames=seen, frames_with_plate=hits, detections=len(ws),
                median_px=round(float(np.median(ws)),1) if ws else 0,
                max_px=round(max(ws),1) if ws else 0)

base=probe('models/plate_detect.pt'); ft=probe('models/plate_detect_ft.pt')
print('baseline   ',base)
print('fine-tuned ',ft)
d=ft['detections']-base['detections']
print(f'\nplate detections: {base["detections"]} -> {ft["detections"]}  ({d:+d})')
print('More detections at 20-35px is the win. If it did NOT improve, the fine-tune')
print('did not help and should not be shipped — say so rather than shipping it.')

## 7. Fit the plate-colour rule

Colour is the part that works at 34 px. Measured over 2,017 EALPR plates, **89%
sit at H 90–120 as one blue population** — which is why the old
light-blue/dark-blue split was wrong and has been merged.

Note what the data cannot do: ~30 red and ~30 orange against 1,792 blue. That
cannot train a balanced classifier — a model fit on it learns "always blue",
scores 90%, and is useless on the rare classes that carry the information. Hence
a hue rule with an explicit `unknown`, not a fitted model.

In [ ]:
!python -m tools.build_plate_colour_set --stage features
!python -m tools.build_plate_colour_set --stage cluster --k 8

## 8. Run the pipeline and write the CSV

In [ ]:
!python -m pipeline.process_video --input samples/street_egypt.mp4 \
    --output-dir data/jobs/colab --stride 3 --plates \
    --plate-model models/plate_detect_ft.pt
!python -m tools.export_plates_csv data/jobs/colab

In [ ]:
import pandas as pd
df=pd.read_csv('data/jobs/colab/plates.csv')
display(df)
ok=(df['agrees_with_class']=='yes').sum(); tot=df['agrees_with_class'].notna().sum()
print(f'plate colour agrees with independently-derived vehicle class: {ok}/{tot}')

## 9. Download the results

Colab wipes the VM on disconnect. Weights are gitignored — do not commit them.

The clip and plate crops are **personal data** (Egypt PDPL 151/2020): clear this
notebook's outputs before sharing it.

In [ ]:
from google.colab import files
import pathlib
for p in ['models/plate_detect_ft.pt','data/jobs/colab/plates.csv',
          'runs/plate/ealpr_ft/results.png']:
    f=pathlib.Path(p)
    if f.exists(): print('downloading',p); files.download(p)
    else: print('missing',p)